In [ ]:

import pandas as pd
from pathlib import Path
from functools import reduce

dossier = Path("../data/totalité_juillet_septembre_2025")
fichiers = sorted(dossier.glob("*.csv"))

print("Nombre de fichiers trouvés :", len(fichiers))

tableaux = []

for fichier in fichiers:

    temp = pd.read_csv(
        fichier,
        sep="\t",
        encoding="cp1252",
        decimal=","
    )

    # Format exact des dates Flowbru
     dates = pd.to_datetime(
    temp.iloc[:, 0].astype(str),
    format="%d.%m.%Y %H:%M:%S",
    errors="coerce"
)
    # Première mesure (brute)
    brute = pd.to_numeric(
        temp.iloc[:, 1],
        errors="coerce"
    )

    if fichier.stem.lower().startswith("pluv"):

        # Les valeurs brutes > 10 mm/5 min
        # correspondent à un étalonnage
        brute = brute.mask(brute > 10)

        mesure = brute.copy()

        # Si une troisième colonne de pluie
        # pré-validée existe, la privilégier
        if len(temp.columns) >= 3:
            nom_col3 = temp.columns[2].lower()

            if "[mm]" in nom_col3 and "vc" not in nom_col3:
                pre_validee = pd.to_numeric(
                    temp.iloc[:, 2],
                    errors="coerce"
                )

                # Utiliser la valeur pré-validée
                # si elle est disponible
                mesure = pre_validee.combine_first(brute)

    else:
        # Stations d'oxygène Q01, Q02, Q03, Q04
        mesure = brute

    propre = pd.DataFrame({
        "date": dates,
        fichier.stem: mesure
    })

    propre = propre.dropna(subset=["date"])

    # Traiter les dates répétées
    # Si les mesures se contredisent :
    # conserver NaN plutôt que choisir au hasard
    def resoudre_doublons(serie):
        valeurs = serie.dropna().unique()

        if len(valeurs) == 1:
            return valeurs[0]

        return float("nan")

    propre = (
        propre.groupby("date", as_index=False)
        .agg({fichier.stem: resoudre_doublons})
    )

    tableaux.append(propre)
    print("Chargé :", fichier.name)

# Fusionner les stations
df = reduce(
    lambda a, b: pd.merge(
        a, b, on="date", how="outer"
    ),
    tableaux
)

# Période juillet à septembre 2025
df = df[
    (df["date"] >= "2025-07-01") &
    (df["date"] < "2025-10-01")
].copy()

df = df.sort_values("date").reset_index(drop=True)

print("\nNombre de lignes :", len(df))
print("Nombre de colonnes :", len(df.columns))

display(df.head())


Problemes apparus: 
1. L'encodage : Python ne reconnaît pas certains caractères accentués.
2. Le séparateur : Python coupe le nom date/time (SER UTC+2,00) au niveau de la virgule, alors que les colonnes des fichers d'oxygène sont séparées par des tabulations.  

Correction:  
  
   df_temp = pd.read_csv(. 
    fichier,  
    sep="\t",  
    encoding="cp1252"
) 

Á savori aussi: dans les fichers des pluviometres il y a plusieurs mesures. 
 
- La première mesure de pluie est une donnée brute, non corrigée.
- Une valeur supérieure à 10 mm en 5 minutes correspond à un étalonnage du pluviomètre, et non à une pluie réelle.
- Les colonnes pré-validées ou validées doivent être privilégiées lorsqu'elles sont disponibles et que leur signification est confirmée.  

Dans ce code: choix d'utiliser la mesure pré-validée lorsqu'elle existe ; sinon utiliser la mesure brute, mais remplacer les valeurs supérieures à 10 mm par une valeur manquante (NaN), jamais par zéro.

In [ ]:
# Compter les valeurs manquantes par station
valeurs_manquantes = df.isna().sum()

print("Valeurs manquantes par station :")
print(valeurs_manquantes)